In [1]:
## import 
import pandas as pd
import numpy as np

In [2]:
trades = pd.read_csv('trades.csv')
reference_prices = pd.read_csv('reference_prices.csv')

In [3]:
trades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 512 entries, 0 to 511
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   trade_id    512 non-null    object 
 1   trade_date  461 non-null    object 
 2   symbol      512 non-null    object 
 3   side        512 non-null    object 
 4   quantity    485 non-null    float64
 5   price       485 non-null    float64
 6   trader      512 non-null    object 
dtypes: float64(2), object(5)
memory usage: 28.1+ KB


In [4]:
reference_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284 entries, 0 to 283
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         284 non-null    object 
 1   symbol       284 non-null    object 
 2   close_price  284 non-null    float64
dtypes: float64(1), object(2)
memory usage: 6.8+ KB


In [5]:
trades.describe(include='all')

,trade_id,trade_date,symbol,side,quantity,price,trader
count,512,461,512,512,485.000000,485.000000,512
unique,500,102,10,2,NaN,NaN,5
top,T1067,2026-01-21,MSFT,SELL,NaN,NaN,J. Byrne
freq,2,13,103,260,NaN,NaN,115
mean,NaN,NaN,NaN,NaN,2422.721649,226.696701,NaN
std,NaN,NaN,NaN,NaN,1432.849087,99.964174,NaN
min,NaN,NaN,NaN,NaN,25.000000,121.940000,NaN
25%,NaN,NaN,NaN,NaN,1097.000000,174.050000,NaN
50%,NaN,NaN,NaN,NaN,2434.000000,188.800000,NaN
75%,NaN,NaN,NaN,NaN,3610.000000,201.760000,NaN


In [6]:
reference_prices.describe(include='all')

,date,symbol,close_price
count,284,284,284.000000
unique,63,5,NaN
top,2026-02-25,AMZN,NaN
freq,5,59,NaN
mean,NaN,NaN,221.982042
std,NaN,NaN,95.983110
min,NaN,NaN,132.600000
25%,NaN,NaN,168.135000
50%,NaN,NaN,194.510000
75%,NaN,NaN,201.152500


## trades table
- duplicate (12 rows, trade_id)
- NaN (trade_date, quantity, price)
## unconsistent between trades and reference_prices
- symbol (10 in trades, 5 in reference_prices)

In [7]:
trades.drop_duplicates()

,trade_id,trade_date,symbol,side,quantity,price,trader
0,T1301,2026-02-17,AAPL,SELL,1892.0,187.07,J. Byrne
1,T1232,2026-02-17,AMZN,BUY,1084.0,203.02,M. Chen
2,T1051,2026-01-07,GOOG,BUY,2286.0,181.77,A. Kowalski
3,T1305,2026-02-06,AMZN,SELL,2458.0,193.44,J. Byrne
4,T1470,NaN,AAPL,BUY,3064.0,191.11,L. O'Sullivan
...,...,...,...,...,...,...,...
505,T1407,2026-01-28,GOOG,BUY,1881.0,173.70,S. Rao
506,T1211,2026-03-24,GOOG,SELL,784.0,175.35,M. Chen
509,T1025,03/03/2026,NVDA,SELL,135.0,128.37,M. Chen
510,T1196,2026-01-12,AMZN,SELL,3802.0,194.91,J. Byrne


In [8]:
trades.symbol.str.upper().unique()

array(['AAPL', 'AMZN', 'GOOG', 'NVDA', 'MSFT'], dtype=object)

In [9]:
reference_prices.symbol.unique()

array(['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA'], dtype=object)

In [10]:
trades.drop_duplicates(inplace=True)
trades['symbol'] = trades.symbol.str.upper() 

trade_dtypes  = {'trade_id':'string',
                 'trade_date':'datetime64[ns]',
                'symbol':'category',
                'side':'category',
                'quantity':'float32',
                'price':'float32',
                'trader':'category'}

trades = trades.astype(trade_dtypes)

In [11]:
symbol_dtype = trades.symbol.dtype

In [12]:
## reference_prices
reference_dtypes = {'date':'datetime64[ns]',
                  'symbol':symbol_dtype,
                  'close_price':'float32'}

reference_prices = reference_prices.astype(reference_dtypes)

In [13]:
## fillna for trades
## for trade_date, drop
## for quantity and price, ffill
trades = trades.dropna(subset=['trade_date'])


In [14]:
trades

,trade_id,trade_date,symbol,side,quantity,price,trader
0,T1301,2026-02-17,AAPL,SELL,1892.0,187.070007,J. Byrne
1,T1232,2026-02-17,AMZN,BUY,1084.0,203.020004,M. Chen
2,T1051,2026-01-07,GOOG,BUY,2286.0,181.770004,A. Kowalski
3,T1305,2026-02-06,AMZN,SELL,2458.0,193.440002,J. Byrne
5,T1162,2026-01-20,GOOG,BUY,4967.0,NaN,M. Chen
...,...,...,...,...,...,...,...
505,T1407,2026-01-28,GOOG,BUY,1881.0,173.699997,S. Rao
506,T1211,2026-03-24,GOOG,SELL,784.0,175.350006,M. Chen
509,T1025,2026-03-03,NVDA,SELL,135.0,128.369995,M. Chen
510,T1196,2026-01-12,AMZN,SELL,3802.0,194.910004,J. Byrne


In [15]:
trades.sort_values('trade_date',inplace=True)

In [16]:
trades.rename(columns={'trade_date':'date'},inplace=True)

In [17]:
trades.set_index('date',inplace=True)

In [18]:
trades['price'] = trades.groupby('symbol')['price'].transform(lambda x: x.interpolate(method='linear'))

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/185379003.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trades['price'] = trades.groupby('symbol')['price'].transform(lambda x: x.interpolate(method='linear'))


In [19]:
trades.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449 entries, 2026-01-02 to 2026-12-03
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   trade_id  449 non-null    string  
 1   symbol    449 non-null    category
 2   side      449 non-null    category
 3   quantity  425 non-null    float32 
 4   price     449 non-null    float32 
 5   trader    449 non-null    category
dtypes: category(3), float32(2), string(1)
memory usage: 12.4 KB


In [20]:
trades[(trades.symbol=='NVDA')&(trades.trader=='J. Byrne')]['quantity'].describe()

count      14.000000
mean     2437.500000
std      1390.641113
min       763.000000
25%      1200.750000
50%      2442.500000
75%      3467.250000
max      4817.000000
Name: quantity, dtype: float64

In [26]:
trades[(trades.symbol=='NVDA')&(trades.trader=='J. Byrne')&(trades.side=='SELL')]['quantity'].describe()

count       3.000000
mean     3267.666748
std      1016.575806
min      2469.000000
25%      2695.500000
50%      2922.000000
75%      3667.000000
max      4412.000000
Name: quantity, dtype: float64

In [21]:
trades.trader.unique()

['J. Byrne', 'L. O'Sullivan', 'A. Kowalski', 'S. Rao', 'M. Chen']
Categories (5, object): ['A. Kowalski', 'J. Byrne', 'L. O'Sullivan', 'M. Chen', 'S. Rao']

In [22]:
trades[(trades.symbol=='NVDA')&(trades.trader=="L. O'Sullivan")]['quantity'].describe()

count       9.000000
mean     1829.777832
std      1407.064453
min       170.000000
25%       690.000000
50%      1891.000000
75%      2666.000000
max      4515.000000
Name: quantity, dtype: float64

In [30]:
trades[(trades.symbol=='NVDA')&(trades.trader=="L. O'Sullivan")&(trades.side=='SELL')]['quantity']

date
2026-01-21    1891.0
2026-03-13     170.0
2026-12-03     241.0
Name: quantity, dtype: float32

In [31]:
## for quantity: groupby symbol, trader, side and fill by mean
trades['quantity'] = trades.groupby(['symbol','trader','side'])['quantity'].transform('mean')

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/1966134422.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trades['quantity'] = trades.groupby(['symbol','trader','side'])['quantity'].transform('mean')


In [32]:
trades.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 449 entries, 2026-01-02 to 2026-12-03
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   trade_id  449 non-null    string  
 1   symbol    449 non-null    category
 2   side      449 non-null    category
 3   quantity  449 non-null    float32 
 4   price     449 non-null    float32 
 5   trader    449 non-null    category
dtypes: category(3), float32(2), string(1)
memory usage: 12.4 KB


In [40]:
%%timeit
mask = (trades.symbol=="NVDA") & (trades.side=='SELL') & (trades.quantity> 1000)
trades[mask]

172 μs ± 2.05 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [44]:
%%timeit
mask = (trades.trader=="J. Byrne") & (trades.index > pd.Timestamp(year=2026,day=1,month=2))
trades[mask]

140 μs ± 24.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [45]:
%%timeit
# mask = (trades.symbol=="NVDA") & (trades.side=='SELL') & (trades.quantity> 1000)
trades.query('symbol=="NVDA" and side=="SELL" and quantity > 1000')

774 μs ± 10.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [46]:
trades.reset_index(inplace=True)

In [48]:
%%timeit
# mask = (trades.trader=="J. Byrne") & (trades.index > pd.Timestamp(year=2026,day=1,month=2))
t = pd.Timestamp(year=2026,day=1,month=2)
trades.query('trader=="J. Byrne" and date > @t')

732 μs ± 80.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [63]:
merge_tabl = trades.merge(reference_prices,on=['date','symbol'],how='left')

In [64]:
merge_tabl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 449 entries, 0 to 448
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         449 non-null    datetime64[ns]
 1   trade_id     449 non-null    string        
 2   symbol       449 non-null    category      
 3   side         449 non-null    category      
 4   quantity     449 non-null    float32       
 5   price        449 non-null    float32       
 6   trader       449 non-null    category      
 7   close_price  388 non-null    float32       
dtypes: category(3), datetime64[ns](1), float32(3), string(1)
memory usage: 14.3 KB


In [65]:
merge_tabl.set_index('date',inplace=True)

In [66]:
merge_tabl['close_price'] = merge_tabl.groupby(['symbol'])['close_price'].transform('mean')

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/3501066568.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merge_tabl['close_price'] = merge_tabl.groupby(['symbol'])['close_price'].transform('mean')


In [69]:
trades.reset_index(inplace=True)
reference_prices.reset_index(inplace=True)

In [70]:
# trades.set_index('date',inplace=True)
trades.sort_values('date',inplace=True)
# reference_prices.set_index('date',inplace=True)
reference_prices.sort_values('date',inplace=True)
pd.merge_asof(trades,reference_prices,on=['date'],by=['symbol'])

,date,trade_id,symbol,side,quantity,price,trader,close_price
0,2026-01-02,T1382,AAPL,BUY,2575.800049,186.380005,J. Byrne,188.630005
1,2026-01-02,T1373,AAPL,BUY,2575.800049,198.080002,J. Byrne,188.630005
2,2026-01-02,T1298,GOOG,SELL,1946.833374,174.990005,J. Byrne,174.800003
3,2026-01-02,T1114,MSFT,SELL,2320.699951,416.000000,L. O'Sullivan,409.839996
4,2026-01-05,T1155,GOOG,SELL,3257.428467,174.880005,A. Kowalski,176.289993
...,...,...,...,...,...,...,...,...
444,2026-11-02,T1479,NVDA,SELL,2500.555664,131.550003,S. Rao,140.369995
445,2026-12-02,T1090,MSFT,BUY,2562.199951,404.899994,M. Chen,399.779999
446,2026-12-02,T1355,AAPL,SELL,1314.142822,191.520004,A. Kowalski,196.360001
447,2026-12-03,T1370,NVDA,BUY,2211.090820,129.190002,J. Byrne,140.369995


In [76]:
merge_tabl = pd.merge_asof(trades,reference_prices,on=['date'],by=['symbol'],direction='nearest')

In [77]:
merge_tabl.head()

,date,trade_id,symbol,side,quantity,price,trader,close_price
0,2026-01-02,T1382,AAPL,BUY,2575.800049,186.380005,J. Byrne,188.630005
1,2026-01-02,T1373,AAPL,BUY,2575.800049,198.080002,J. Byrne,188.630005
2,2026-01-02,T1298,GOOG,SELL,1946.833374,174.990005,J. Byrne,174.800003
3,2026-01-02,T1114,MSFT,SELL,2320.699951,416.000000,L. O'Sullivan,409.839996
4,2026-01-05,T1155,GOOG,SELL,3257.428467,174.880005,A. Kowalski,176.289993


In [78]:
merge_tabl['notional'] = merge_tabl['quantity'] * merge_tabl['price'] 

In [79]:
merge_tabl['ref_notional'] = merge_tabl['quantity'] * merge_tabl['close_price'] 

In [80]:
merge_tabl['diff_notional'] = np.abs(merge_tabl['ref_notional'] - merge_tabl['notional'])

In [81]:
merge_tabl.nlargest(5,columns='diff_notional')

,date,trade_id,symbol,side,quantity,price,trader,close_price,notional,ref_notional,diff_notional
421,2026-03-30,T1430,MSFT,BUY,3346.000000,420.160004,L. O'Sullivan,398.890015,1.405855e+06,1.334686e+06,71169.37500
406,2026-03-26,T1428,MSFT,SELL,2883.500000,414.250000,M. Chen,397.239990,1.194490e+06,1.145442e+06,49048.37500
321,2026-03-10,T1234,AMZN,BUY,2302.909180,186.119995,J. Byrne,206.580002,4.286174e+05,4.757350e+05,47117.53125
318,2026-03-10,T1076,GOOG,SELL,3257.428467,181.809998,A. Kowalski,167.639999,5.922331e+05,5.460753e+05,46157.75000
409,2026-03-26,T1023,MSFT,SELL,2320.699951,417.029999,L. O'Sullivan,397.239990,9.678015e+05,9.218748e+05,45926.68750


In [88]:
trades.groupby('symbol',observed=True)['quantity'].transform('sum')

0      238171.296875
1      238171.296875
2      239990.500000
3      244788.093750
4      239990.500000
           ...      
444    167993.562500
445    244788.093750
446    238171.296875
447    167993.562500
448    167993.562500
Name: quantity, Length: 449, dtype: float32

In [92]:
# 14. Total notional value (`quantity * price`) per trader, sorted descending.
merge_tabl.groupby(['trader'])['notional'].agg('sum').sort_values(ascending=False)

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/3632647512.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merge_tabl.groupby(['trader'])['notional'].agg('sum').sort_values(ascending=False)


trader
J. Byrne         56846076.0
M. Chen          52884448.0
L. O'Sullivan    47551764.0
A. Kowalski      47469892.0
S. Rao           43890084.0
Name: notional, dtype: float32

In [108]:
# For each symbol, the top 3 traders by total notional value. (This is the
#     "top-N per group" pattern — think `groupby` + `apply`/`nlargest`, and be
#     ready to explain the performance trade-off vs a fully vectorized
#     approach.)
# merge_tabl['trader_symbol_notional'] = merge_tabl.groupby(['trader','symbol'])['notional'].transform('sum')
# merge_tabl.groupby('symbol')['trader_symbol_notional'].nlargest(3)
merge_tabl.groupby(['symbol','trader'])['notional'].agg('sum').groupby(level=0,group_keys=False).nlargest(3)

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/1915036141.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merge_tabl.groupby(['symbol','trader'])['notional'].agg('sum').groupby(level=0,group_keys=False).nlargest(3)


symbol  trader       
AAPL    L. O'Sullivan    12926865.0
        J. Byrne         10407105.0
        S. Rao            8708362.0
AMZN    J. Byrne         13668827.0
        S. Rao            8221035.0
        L. O'Sullivan     7024692.5
GOOG    M. Chen           9973489.0
        J. Byrne          9939921.0
        A. Kowalski       9549352.0
MSFT    M. Chen          24481406.0
        A. Kowalski      24398010.0
        L. O'Sullivan    18787420.0
NVDA    M. Chen           5961350.0
        J. Byrne          5886788.5
        S. Rao            5128978.5
Name: notional, dtype: float32

In [109]:
# 16. Average trade size (`quantity`) per symbol **per side** (BUY vs SELL) —
#     i.e. group by two keys.
trades.groupby(['symbol','side'])['quantity'].agg('mean').reset_index()

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/2296346055.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trades.groupby(['symbol','side'])['quantity'].agg('mean').reset_index()


,symbol,side,quantity
0,AAPL,BUY,2736.086182
1,AAPL,SELL,2383.849365
2,AMZN,BUY,2245.797607
3,AMZN,SELL,2708.618652
4,GOOG,BUY,2473.963379
5,GOOG,SELL,2177.408691
6,MSFT,BUY,2412.089844
7,MSFT,SELL,2637.410400
8,NVDA,BUY,2194.523926
9,NVDA,SELL,2230.104492


In [ ]:
# 17. Using `.transform()`, add a column to the trades dataframe showing each
#     trade's quantity as a percentage of that symbol's total daily volume.
#     (Tests: do you reach for `apply` when `transform` is the right, faster
#     tool?)

trades['daily_qty'] = trades.groupby(['date','symbol'])['quantity'].transform('sum')
trades['daily_qty_percent'] = trades['daily_qty']trades['daily_qty']

In [113]:
trades['daily_qty_percent']  = trades.groupby(['date','symbol'])['quantity'].transform(lambda x: x/sum(x)* 100)

/var/folders/vc/5v67y27d32dbbrzlq4rdl5y80000gn/T/ipykernel_14439/3782962600.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trades['daily_qty_percent']  = trades.groupby(['date','symbol'])['quantity'].transform(lambda x: x/sum(x)* 100)


In [118]:
# 18. Resample: what was the total daily notional traded, across all symbols,
#     for each calendar day? (Tip: needs a datetime index or `.resample()` /
#     `.groupby(pd.Grouper(...))`.)

trades['notional'] = trades['price'] * trades['quantity']
trades.groupby('date').agg(daily_notional=('notional','sum')).resample('1D').ffill()

,daily_notional
date,
2026-01-02,2296379.500
2026-01-03,2296379.500
2026-01-04,2296379.500
2026-01-05,3716020.250
2026-01-06,3537119.500
...,...
2026-11-29,2224444.750
2026-11-30,2224444.750
2026-12-01,2224444.750


In [119]:
trades.groupby('date').agg(daily_notional=('notional','sum'))

,daily_notional
date,
2026-01-02,2.296380e+06
2026-01-05,3.716020e+06
2026-01-06,3.537120e+06
2026-01-07,2.979720e+06
2026-01-08,2.415246e+06
...,...
2026-09-02,1.373485e+06
2026-10-03,5.093784e+05
2026-11-02,2.224445e+06


In [122]:
trades.groupby('date').agg(daily_notional=('notional','sum')).resample('D',label='right').ffill()

,daily_notional
date,
2026-01-02,2296379.500
2026-01-03,2296379.500
2026-01-04,2296379.500
2026-01-05,3716020.250
2026-01-06,3537119.500
...,...
2026-11-29,2224444.750
2026-11-30,2224444.750
2026-12-01,2224444.750


In [124]:
trades.groupby('date').agg(daily_notional=('notional','sum')).resample('D',label='left').ffill()

,daily_notional
date,
2026-01-02,2296379.500
2026-01-03,2296379.500
2026-01-04,2296379.500
2026-01-05,3716020.250
2026-01-06,3537119.500
...,...
2026-11-29,2224444.750
2026-11-30,2224444.750
2026-12-01,2224444.750


# Big data

In [125]:
big_trades = pd.read_csv('big_trades.csv',nrows=100)

In [129]:
import pandas as pd, time

def timed_read(**kwargs):
    t0 = time.perf_counter()
    df = pd.read_csv("big_trades.csv", **kwargs)
    print(f"{kwargs}: {time.perf_counter()-t0:.2f}s")
    return df

df1 = timed_read()                      # default C engine
df2 = timed_read(engine="pyarrow")      # multi-threaded Arrow parser

{}: 0.86s


ImportError: Missing optional dependency 'pyarrow'.  Use pip or conda to install pyarrow.

In [130]:
!which pip

/opt/homebrew/bin/pip
